# Sentinel-2 acquisition planner

Short driver notebook for the reusable Sentinel-2 Python module.

Edit only the paths in the configuration cell, then run the notebook top-to-bottom.

In [ ]:
%pip install -q requests beautifulsoup4 shapely pandas geopandas pyogrio matplotlib

In [ ]:
from pathlib import Path
from datetime import datetime, timedelta, timezone
import importlib.util
import sys

# ------------------------------------------------------------
# INPUT PATHS
# ------------------------------------------------------------

repo_root = Path.home() / "mystorage" / "fire-school"

candidates = [
    repo_root / "data" / "aoi" / "ALB_WF001_AOI.shp",
    Path.cwd() / "data" / "aoi" / "ALB_WF001_AOI.shp",
    Path.cwd().parent / "data" / "aoi" / "ALB_WF001_AOI.shp",
]

AOI = next((p for p in candidates if p.exists()), None)

if AOI is None:
    raise FileNotFoundError(
        "Canonical AOI file data/aoi/ALB_WF001_AOI.shp was not found. "
        "Run git pull in the fire-school repository."
    )

print("Using:", AOI)

# The companion module can be found from the repository or notebooks folder.
module_candidates = [
    repo_root / "notebooks" / "sentinel2_acquisition.py",
    Path.cwd() / "notebooks" / "sentinel2_acquisition.py",
    Path.cwd().parent / "notebooks" / "sentinel2_acquisition.py",
    Path.cwd() / "sentinel2_acquisition.py",
]

MODULE_FILE = next((p for p in module_candidates if p.exists()), None)

if MODULE_FILE is None:
    raise FileNotFoundError(
        "Companion module notebooks/sentinel2_acquisition.py was not found. "
        "Run git pull in the fire-school repository."
    )

# Local timezone used only for plot titles.
#TIMEZONE = "Europe/Tirane"
TIMEZONE = "UTC" # UTC stands for Coordinated Universal Time, Albania summertime + 2 hours

# Number of days to search.
DAYS = 14

print("Module:", MODULE_FILE)
print("AOI:   ", AOI)
print("Days:  ", DAYS)
print("TZ:    ", TIMEZONE)

In [ ]:
if not MODULE_FILE.exists():
    raise FileNotFoundError(
        f"Python module not found: {MODULE_FILE}"
    )

module_name = "sentinel2_acquisition_module"

spec = importlib.util.spec_from_file_location(
    module_name,
    MODULE_FILE,
)

s2mod = importlib.util.module_from_spec(spec)

# Important for dataclasses / postponed annotations
sys.modules[module_name] = s2mod

spec.loader.exec_module(s2mod)

find_sentinel2_acquisitions = (
    s2mod.find_sentinel2_acquisitions
)

plot_sentinel2_overpasses = (
    s2mod.plot_sentinel2_overpasses
)

print("Sentinel-2 functions loaded successfully.")

## List planned Sentinel-2 acquisitions for the next week

In [ ]:
sentinel2 = find_sentinel2_acquisitions(
    aoi_path=AOI,
    days=DAYS,
)

In [ ]:
# Clean table

if sentinel2.empty:
    print("No planned Sentinel-2 acquisitions found.")
else:
    display(
        sentinel2[
            [
                "satellite",
                "acquisition_id",
                "start_utc",
                "end_utc",
            ]
        ]
    )

## Plot one figure per planned overpass

In [ ]:
sentinel2_plotted, figures = plot_sentinel2_overpasses(
    aoi_path=AOI,
    days=DAYS,
    timezone_name=TIMEZONE,
)

## Search earlier than current time

In [ ]:
# make the start time based on days before today
days_before_now=21

start_time = (
    datetime.now(timezone.utc)
    - timedelta(days=days_before_now)
)
print(start_time)

In [ ]:
sentinel2_before = find_sentinel2_acquisitions(
    aoi_path=AOI,
    start=start_time,
    days=21,
)

sentinel2_before

## Plot the previous days

In [ ]:
sentinel2_before_plotted, figures_last_week = (
    plot_sentinel2_overpasses(
        aoi_path=AOI,
        start=start_time,
        days=21,
        timezone_name=TIMEZONE,
    )
)

## Optional: save the current results to CSV

In [ ]:
OUTPUT_CSV = Path(
    "mystorage/S2-aquisitions-ALB-WF1.csv"
)

# change the to the table you want to save e.g. (sentinel2_before_plotted, or sentinel2)
sentinel2_before_plotted.to_csv(
    OUTPUT_CSV,
    index=False,
)

print("Saved:", OUTPUT_CSV)